# Transfer Bayesian Optimization 使用ガイド

連続変数・Mixed変数の転移ベイズ最適化を試します。`q=1`、Sequential greedy `q>1`、Joint batch `q>1` を扱います。候補は提案のみで、目的関数の評価は利用者側で行います。

In [ ]:
import torch
from botorch.models import SingleTaskGP
from robotorchan.optim.transfer.taf_sequential import propose_taf_candidate
from robotorchan.optim.transfer.rgpe_sequential import propose_rgpe_candidate

torch.set_default_dtype(torch.double)
bounds = torch.tensor([[0.0], [1.0]])
def target_objective(x):
    return -((x - 0.75) ** 2)

source_x = torch.linspace(0, 1, 6).unsqueeze(-1)
source_y = -((source_x - 0.7) ** 2)
source_model = SingleTaskGP(source_x, source_y)
target_x = torch.tensor([[0.0], [0.3], [1.0]])
target_y = target_objective(target_x)

## Warm-startによる初期点選択

ソースの観測値はターゲットの観測値ではありません。候補点をターゲット目的関数で評価してから学習に使います。

In [ ]:
from robotorchan.optim.transfer.warm_start import select_warm_start_points

initial_x = select_warm_start_points(
    bounds, 3, source_x=source_x, source_y=source_y, seed=0,
)
initial_y = target_objective(initial_x)
print('Warm-start target observations:', initial_x, initial_y)

## TAFによる逐次候補提案

重みはソース・ターゲットの順で指定します。評価と観測追加は利用者側で行います。

In [ ]:
weights = torch.tensor([0.5, 0.5])
for iteration in range(2):
    candidate = propose_taf_candidate(
        [source_model], target_x, target_y, bounds, weights,
        raw_samples=16, num_restarts=2, seed=iteration,
    )
    observation = target_objective(candidate)
    target_x = torch.cat([target_x, candidate])
    target_y = torch.cat([target_y, observation])
    print(f'TAF iteration={iteration + 1}, x={candidate.item():.4f}, best={target_y.max().item():.4f}')

## RGPEによる候補提案

RGPEはターゲットのLOO予測を得るため複数回GPを学習します。TAFと異なり、重みをランキング損失から推定します。

In [ ]:
candidate, learned_weights = propose_rgpe_candidate(
    [source_model], target_x, target_y, bounds,
    n_bootstrap=8, raw_samples=16, num_restarts=2, seed=0,
)
print('RGPE candidate:', candidate)
print('Source / target weights:', learned_weights)

## 評価と注意点

同一初期観測から手法間の学習曲線を比較するには `examples/benchmarks/transfer_bo.py` を参照してください。負の転移はソースとターゲットの応答面が異なるときに起こり得ます。単一試行の結果から性能の優劣は判断できません。

## SequentialとJointのバッチ提案

`sequential` は仮想観測を使って1点ずつ提案します。`joint` は候補集合を同時に最適化します。両者の候補は実際に評価するまでターゲットの観測値ではありません。

In [ ]:
for strategy in ("sequential", "joint"):
    batch = propose_taf_candidate(
        [source_model], target_x, target_y, bounds, weights,
        q=2, batch_strategy=strategy, mc_samples=32,
        raw_samples=16, num_restarts=2, seed=0,
    )
    print(strategy, "TAF candidates:", batch)


## Mixed変数：カテゴリと連続変数

2列目をカテゴリ変数（0または1）として扱います。`cat_dims` と `fixed_features_list` はセットで指定します。ここではカテゴリを含むターゲットの目的関数を定義し、TAF・RGPEの両方で候補を提案します。

In [ ]:
mixed_bounds = torch.tensor([[0.0, 0.0], [1.0, 1.0]])
mixed_x = torch.tensor([[0.0, 0.0], [0.5, 1.0], [1.0, 0.0], [0.2, 1.0]])
def mixed_objective(x):
    return -((x[:, :1] - 0.7) ** 2) + 0.1 * x[:, 1:2]

mixed_y = mixed_objective(mixed_x)
categories = [{1: 0.0}, {1: 1.0}]
for strategy in ("sequential", "joint"):
    taf_batch = propose_taf_candidate(
        [], mixed_x, mixed_y, mixed_bounds, torch.ones(1),
        cat_dims=[1], fixed_features_list=categories,
        q=2, batch_strategy=strategy, mc_samples=32,
        raw_samples=16, num_restarts=2, seed=0,
    )
    print(strategy, "Mixed TAF:", taf_batch)

rgpe_batch, rgpe_weights = propose_rgpe_candidate(
    [], mixed_x, mixed_y, mixed_bounds,
    cat_dims=[1], fixed_features_list=categories,
    q=2, batch_strategy="joint", n_bootstrap=8,
    mc_samples=32, raw_samples=16, num_restarts=2, seed=0,
)
print("Mixed RGPE joint:", rgpe_batch, rgpe_weights)
